# Day 14: Long-Term Memory and Runtime Context

On Day 7 we gave our chatbot memory with a checkpointer. But that memory belongs to **one conversation** (one thread). Start a new chat, and the bot has no idea who you are.

Think about a good local doctor. Each visit is a separate conversation. But the doctor keeps a **file** on you: allergies, past problems, what worked last time. When you walk in for a new visit, they glance at your file first. That file is long-term memory.

So now we have two kinds:

- **Short-term memory (checkpointer, Day 7)**: the message history of one conversation. "What did I just say?"
- **Long-term memory (Store, today)**: facts about the user, across all conversations. "She's vegetarian and prefers Hindi."

We'll also learn about **runtime context**: a clean way to pass things like "which user is this?" into the graph without mixing them into the conversation state.

By the end of today you'll have a chatbot that remembers you even in a brand new chat.

In [ ]:
%pip install -q -U langgraph langchain-openai python-dotenv

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from IPython.display import Image, display

In [ ]:
load_dotenv(find_dotenv(usecwd=True))
assert os.getenv("OPENAI_API_KEY"), "Add OPENAI_API_KEY to the .env file in the repo root (see Day 4)."
llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL", "gpt-4.1-nano"), temperature=0)

In [ ]:
def show_graph(app):
    try:
        display(Image(app.get_graph().draw_mermaid_png()))
    except Exception:
        print(app.get_graph().draw_mermaid())

## 1. The Store: a filing cabinet

A **Store** is a simple place to keep small JSON documents. Every document lives in a **namespace**, which is a tuple that works like a folder path, and has a **key**, like a file name.

```
("users", "u42", "memories")
     |-- "food"  ->  {"fact": "is vegetarian"}
     |-- "lang"  ->  {"fact": "loves Python"}
```

So `("users", "u42", "memories")` is like the folder `users/u42/memories/`, and inside it there are files called `food` and `lang`.

No LLM in this section, just the basic operations.

In [ ]:
from langgraph.store.memory import InMemoryStore

In [ ]:
store = InMemoryStore()
folder = ("users", "u42", "memories")

In [ ]:
# put(folder, key, value) saves a document
store.put(folder, "food", {"fact": "is vegetarian"})
store.put(folder, "lang", {"fact": "loves Python"})

In [ ]:
# get(folder, key) reads one document back
item = store.get(folder, "food")
print(item.key, "->", item.value)

In [ ]:
# search(folder) lists everything in a folder
for item in store.search(folder):
    print(item.key, "->", item.value["fact"])

### Predict before you run

We'll `put` a new value with the **same key** `"food"`. Will there now be two "food" entries, or one?

<details>
<summary>Click to see the answer</summary>

One. Using the same key **overwrites** the old value, just like saving a file with the same name.
</details>

In [ ]:
store.put(folder, "food", {"fact": "is vegan now"})

print(store.get(folder, "food").value)
print("Total entries:", len(store.search(folder)))

In [ ]:
store.delete(folder, "lang")
print([item.key for item in store.search(folder)])

## 2. Runtime context: "who is this?"

Some information isn't part of the conversation, but the graph still needs it: which user is talking, which language they want, maybe a database connection. Putting these into the chat state would be messy, they're settings for **this run**, not messages.

Think of it like the token number you get at a bank counter. It isn't part of what you discuss with the clerk, but it tells the clerk whose file to pull out.

You describe the context with a small dataclass, tell the graph about it with `context_schema=`, and pass it in with `invoke(..., context=...)`. Inside a node, add a `runtime` argument and read `runtime.context`.

In [ ]:
from dataclasses import dataclass
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.runtime import Runtime

In [ ]:
@dataclass
class Context:
    user_id: str
    language: str = "English"

In [ ]:
class GreetState(TypedDict):
    greeting: str

In [ ]:
def greet(state: GreetState, runtime: Runtime[Context]) -> dict:
    # The extra "runtime" argument is filled in by LangGraph automatically
    user = runtime.context.user_id
    language = runtime.context.language
    return {"greeting": f"[{language}] Hello, user {user}!"}

In [ ]:
builder = StateGraph(GreetState, context_schema=Context)

In [ ]:
builder.add_node("greet", greet)

In [ ]:
builder.add_edge(START, "greet")
builder.add_edge("greet", END)

In [ ]:
greet_app = builder.compile()

In [ ]:
print(greet_app.invoke({}, context=Context(user_id="u42")))
print(greet_app.invoke({}, context=Context(user_id="u7", language="Marathi")))

## 3. Using the Store inside a graph

Compile the graph with `store=store`, and every node can reach it through `runtime.store`. Combine that with `runtime.context.user_id` and each node can open **that user's** folder.

This bot reads the user's facts before answering, like the doctor glancing at your file.

In [ ]:
from langgraph.graph import MessagesState
from langchain_core.messages import SystemMessage

In [ ]:
def personal_bot(state: MessagesState, runtime: Runtime[Context]) -> dict:
    folder = ("users", runtime.context.user_id, "memories")
    facts = [item.value["fact"] for item in runtime.store.search(folder)]

    if facts:
        known = ", ".join(facts)
    else:
        known = "nothing yet"

    instructions = SystemMessage(content=f"You are a helpful assistant. What you know about this user: {known}.")
    reply = llm.invoke([instructions] + state["messages"])
    return {"messages": [reply]}

In [ ]:
builder = StateGraph(MessagesState, context_schema=Context)

In [ ]:
builder.add_node("bot", personal_bot)

In [ ]:
builder.add_edge(START, "bot")
builder.add_edge("bot", END)

In [ ]:
personal_app = builder.compile(store=store)

### Predict before you run

We'll ask the same question, "Suggest a dinner for tonight", as user `u42` (who we saved as vegan) and as a stranger. Will the answers differ?

<details>
<summary>Click to see the answer</summary>

They should. For `u42` the bot sees "is vegan now" in its instructions and should suggest something vegan. For the stranger it knows nothing, so it can suggest anything.
</details>

In [ ]:
question = {"messages": [("user", "Suggest a dinner for tonight.")]}

result = personal_app.invoke(question, context=Context(user_id="u42"))
print("For u42:     ", result["messages"][-1].content)
print()

result = personal_app.invoke(question, context=Context(user_id="stranger"))
print("For stranger:", result["messages"][-1].content)

## 4. A chatbot that remembers you in every new chat

So far *we* wrote the facts into the store by hand. A real assistant should notice facts while you chat and save them itself.

Our graph gets two nodes:

- `chat` reads the user's facts and replies
- `remember` asks the LLM "did the user just tell us anything worth remembering?", and saves any new facts to the store

```
  START -> chat (reads memories) -> remember (saves new facts) -> END
```

And we'll use **both** kinds of memory: a checkpointer for the conversation, and a store for the long-term facts.

In [ ]:
import uuid
from pydantic import BaseModel, Field
from langgraph.checkpoint.memory import InMemorySaver

In [ ]:
class NewFacts(BaseModel):
    facts: list[str] = Field(
        description="new long-term facts about the USER, such as name, job, likes or plans. Empty if there are none."
    )

In [ ]:
fact_extractor = llm.with_structured_output(NewFacts)
memory_store = InMemoryStore()

In [ ]:
def chat(state: MessagesState, runtime: Runtime[Context]) -> dict:
    folder = ("users", runtime.context.user_id, "memories")
    facts = [item.value["fact"] for item in runtime.store.search(folder)]

    instructions = SystemMessage(content=(
        f"You are a friendly assistant. Known facts about the user: {facts or 'none'}. Keep replies short."
    ))
    return {"messages": [llm.invoke([instructions] + state["messages"])]}

In [ ]:
def remember(state: MessagesState, runtime: Runtime[Context]) -> dict:
    # Look only at the user's latest message
    user_messages = [m for m in state["messages"] if m.type == "human"]
    latest = user_messages[-1].content

    found = fact_extractor.invoke(f"Extract long-term facts about the user from this message: {latest}")

    folder = ("users", runtime.context.user_id, "memories")
    for fact in found.facts:
        # A random key, so every fact gets its own entry
        runtime.store.put(folder, str(uuid.uuid4()), {"fact": fact})
        print("Saved:", fact)

    return {}      # nothing to change in the conversation itself

In [ ]:
builder = StateGraph(MessagesState, context_schema=Context)

In [ ]:
builder.add_sequence([chat, remember])

In [ ]:
builder.add_edge(START, "chat")
builder.add_edge("remember", END)

In [ ]:
memory_bot = builder.compile(checkpointer=InMemorySaver(), store=memory_store)
show_graph(memory_bot)

In [ ]:
me = Context(user_id="learner-1")

# Conversation 1, on thread A
memory_bot.invoke(
    {"messages": [("user", "Hi! I'm Neha, a data engineer from Pune, and I love masala chai.")]},
    {"configurable": {"thread_id": "thread-A"}},
    context=me,
)

Now the real test. A **completely new thread**, so there's no message history at all. Can the bot still tell us what it knows?

In [ ]:
result = memory_bot.invoke(
    {"messages": [("user", "What do you know about me? And suggest a drink.")]},
    {"configurable": {"thread_id": "thread-B"}},
    context=me,
)
print(result["messages"][-1].content)

In [ ]:
print("Everything saved for learner-1:")
for item in memory_store.search(("users", "learner-1", "memories")):
    print("-", item.value["fact"])

## 5. Finding the right memories by meaning

After a few months, a user might have hundreds of saved facts. You don't want to stuff all of them into every prompt, only the ones that matter for this question.

Remember embeddings from yesterday? A Store can use them too. Give it an `index` with an embedding model, and then `search(folder, query="...")` returns the memories closest **in meaning** to your query.

In [ ]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

smart_store = InMemoryStore(
    index={"embed": embeddings, "dims": 1536, "fields": ["fact"]}
)

In [ ]:
folder = ("users", "u1", "memories")

facts = [
    "Allergic to peanuts",
    "Works as a data scientist at a bank",
    "Has a pet dog named Bruno",
    "Prefers South Indian breakfast",
    "Is learning LangGraph in the evenings",
]

for number, fact in enumerate(facts):
    smart_store.put(folder, f"m{number}", {"fact": fact})

### Predict before you run

For the query "what should I cook for them?", which two facts do you expect at the top?

<details>
<summary>Click to see the answer</summary>

Most likely "Allergic to peanuts" and "Prefers South Indian breakfast", the two that are about food. The scores show how close each one is. Your exact numbers may differ a little.
</details>

In [ ]:
for query in ["what should I cook for them?", "tell me about their job"]:
    print("Query:", query)
    for hit in smart_store.search(folder, query=query, limit=2):
        print(f"   {hit.score:.2f}  {hit.value['fact']}")
    print()

For production, use `PostgresStore` from the `langgraph-checkpoint-postgres` package. The methods are exactly the same.

## 6. Common mistakes

**Mistake 1: Forgetting `store=` when compiling.**
If you compile without a store, `runtime.store` is `None`, and the first `.search()` fails.

In [ ]:
builder = StateGraph(MessagesState, context_schema=Context)

In [ ]:
builder.add_node("bot", personal_bot)

In [ ]:
builder.add_edge(START, "bot")
builder.add_edge("bot", END)

In [ ]:
no_store_app = builder.compile()          # forgot store=store

In [ ]:
try:
    no_store_app.invoke({"messages": [("user", "hi")]}, context=Context(user_id="u42"))
except AttributeError as error:
    print("AttributeError:", error)

**Fix:** `builder.compile(store=store)`.

**Mistake 2: Forgetting to pass `context=`.**
If a node reads `runtime.context.user_id` and you didn't pass a context, there's nothing to read. Always pass `context=Context(...)` when the graph has a `context_schema`.

**Mistake 3: Putting the user id in the chat messages instead of the context.**
"Hi, I'm user 42" in a message can be faked by anyone typing. The context comes from **your** code (for example, from the logged-in session), so the user can't change it. Keep identity in the context.

**Mistake 4: One shared folder for everyone.**
If every user's facts go into `("memories",)`, everyone's facts get mixed up, and one user might see another's private details. Always include the user id in the namespace.

**Mistake 5: Saving everything.**
Not every sentence is worth remembering. "I'm tired today" is not a long-term fact. The extractor's description ("long-term facts... empty if there are none") is what keeps the store clean, so write it carefully.

## 7. Practice

**Exercise 1.** Write two helper functions using the folder `("prefs", user_id)`:
- `save_pref(store, user_id, key, value)` saves one preference
- `get_prefs(store, user_id)` returns all of them as a normal dictionary

Save `theme = dark` and `lang = Hindi`, then read them back.

**Exercise 2.** Make the `chat` node reply in the language given in the context. Run it for `learner-1` with `language="Marathi"`.

In [ ]:
# Solution 1
def save_pref(store, user_id: str, key: str, value: str) -> None:
    store.put(("prefs", user_id), key, {"value": value})

In [ ]:
def get_prefs(store, user_id: str) -> dict:
    prefs = {}
    for item in store.search(("prefs", user_id)):
        prefs[item.key] = item.value["value"]
    return prefs

In [ ]:
prefs_store = InMemoryStore()

save_pref(prefs_store, "u9", "theme", "dark")
save_pref(prefs_store, "u9", "lang", "Hindi")

print(get_prefs(prefs_store, "u9"))
assert get_prefs(prefs_store, "u9") == {"theme": "dark", "lang": "Hindi"}

In [ ]:
# Solution 2
def chat_in_language(state: MessagesState, runtime: Runtime[Context]) -> dict:
    folder = ("users", runtime.context.user_id, "memories")
    facts = [item.value["fact"] for item in runtime.store.search(folder)]

    instructions = SystemMessage(content=(
        f"Known facts about the user: {facts or 'none'}. "
        f"Always reply in {runtime.context.language}. Keep it short."
    ))
    return {"messages": [llm.invoke([instructions] + state["messages"])]}

In [ ]:
builder = StateGraph(MessagesState, context_schema=Context)

In [ ]:
builder.add_node("chat", chat_in_language)

In [ ]:
builder.add_edge(START, "chat")
builder.add_edge("chat", END)

In [ ]:
language_bot = builder.compile(store=memory_store)

In [ ]:
result = language_bot.invoke(
    {"messages": [("user", "Suggest a drink for me.")]},
    context=Context(user_id="learner-1", language="Marathi"),
)
print(result["messages"][-1].content)

## Wrapping up

The **Store** is long-term memory across conversations: `put`, `get`, `search` and `delete`, organised by namespace folders like `("users", user_id, "memories")`. **Runtime context** carries per-run settings like the user id, through `context_schema=` and `invoke(..., context=...)`, and nodes read both through the `runtime` argument.

The pattern to remember: **read** the user's memories before answering, and **extract and save** new facts after. Add an embedding index when you have lots of memories, so you only pull out the relevant ones.

Tomorrow is about getting your graphs ready for the real world: retries when APIs fail, caching to save money, graceful error handling, testing, and deployment.